# Activity 2 Solution: pandas and Polars Quick Drills

Instructor reference. Do not distribute before the debrief.


In [ ]:
from pathlib import Path
import pandas as pd
import polars as pl

DATA_PATH = Path("data/taxi_trip_review.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Make sure you are running this notebook from the project root "
        "and that the data/taxi_trip_review.csv file exists."
    )

print("Data file found:", DATA_PATH)


## Dataset Contract

For this drill, focus on `trip_distance`, `fare_amount`, and `payment_type`. One row has `abc` in `fare_amount`, so the load step treats that value as missing. The goal is basic DataFrame practice, not a cleaning pipeline.


## Part 1: pandas Practice

Load, inspect, select, filter, flag, and summarize.


In [ ]:
pdf = pd.read_csv(DATA_PATH, na_values=["abc"])
pdf.head()


In [ ]:
pdf.info()


In [ ]:
pdf.describe()


In [ ]:
pdf_small = pdf[["trip_distance", "fare_amount", "payment_type"]]
pdf_small.head()


In [ ]:
pdf_filtered = pdf_small[pdf_small["trip_distance"] > 2.0]
pdf_filtered.head()


In [ ]:
pdf_flagged = pdf_filtered.copy()
pdf_flagged["is_long_trip"] = pdf_flagged["trip_distance"] > 3.0
pdf_flagged[["trip_distance", "is_long_trip"]].head()


In [ ]:
pdf_payment_summary = (
    pdf_flagged.groupby("payment_type", as_index=False)
    .agg(mean_fare_amount=("fare_amount", "mean"))
    .sort_values("payment_type")
)

pdf_payment_summary


## Part 2: Polars Practice

Now repeat the same ideas in Polars. The syntax is different, but the workflow is similar.


In [ ]:
pldf = pl.read_csv(DATA_PATH, null_values=["abc"])
pldf.head()


In [ ]:
print(pldf.schema)


In [ ]:
pldf_small = pldf.select(["trip_distance", "fare_amount", "payment_type"])
pldf_small.head()


In [ ]:
pldf_filtered = pldf_small.filter(pl.col("trip_distance") > 2.0)
pldf_filtered.head()


In [ ]:
pldf_flagged = pldf_filtered.with_columns(
    (pl.col("trip_distance") > 3.0).alias("is_long_trip")
)

pldf_flagged.select(["trip_distance", "is_long_trip"]).head()


In [ ]:
pldf_payment_summary = (
    pldf_flagged.group_by("payment_type")
    .agg(pl.col("fare_amount").mean().alias("mean_fare_amount"))
    .sort("payment_type")
)

pldf_payment_summary


## Final Comparison Checkpoint

These checks confirm that the pandas and Polars drills match on the basic requirements.


In [ ]:
print("Pandas filtered rows:", len(pdf_filtered))
print("Polars filtered rows:", pldf_filtered.height)

assert len(pdf_filtered) == pldf_filtered.height
assert "is_long_trip" in pdf_flagged.columns
assert "is_long_trip" in pldf_flagged.columns

print("Basic checks passed.")
